In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List


c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
print("""
RAG (Retrival-Augmented Generation) Architecture::
      
      1. Document Loading: load documents from various sources
      2. Document splitting:Break document into smallar sources
      3. Embedding Generation: Convert chunks into vector representations
      4. Vectorstrore: store the vectors in a vector database
      5. Query Processing: convert user query to embedding
      6. similarity search: Find relevant chunks from vector store
      7. context Augmentation: combine retrived chunks with user query
      8. Response generation: LLM generates answer using the context

""")


RAG (Retrival-Augmented Generation) Architecture::

      1. Document Loading: load documents from various sources
      2. Document splitting:Break document into smallar sources
      3. Embedding Generation: Convert chunks into vector representations
      4. Vectorstrore: store the vectors in a vector database
      5. Query Processing: convert user query to embedding
      6. similarity search: Find relevant chunks from vector store
      7. context Augmentation: combine retrived chunks with user query
      8. Response generation: LLM generates answer using the context




### Document loading

In [4]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

loader = DirectoryLoader(r"C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\text_files", 
                         glob="**/*.txt", 
                         show_progress=True, 
                         loader_cls=TextLoader,
                         loader_kwargs={"encoding": "utf-8"}
                         )
documents  = loader.load()
print(f"Number of documents loaded: {len(documents)}")


100%|██████████| 4/4 [00:00<00:00, 990.68it/s]

Number of documents loaded: 4


In [5]:
documents

[Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\Java.txt'}, page_content='Java is a versatile programming language. It is used for building enterprise applications, mobile apps, and large-scale systems.'),
 Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\JavaScript.txt'}, page_content='JavaScript is a popular programming language for web development. It is used to create interactive and dynamic web pages.'),
 Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.[1] Within a subdiscipline of machine learn

### Document splitting

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks  = text_splitter.split_documents(documents)
print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 28


In [7]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[ 6.76569045e-02  6.34959415e-02  4.87130694e-02  7.93049410e-02
   3.74480300e-02  2.65279389e-03  3.93749736e-02 -7.09845126e-03
   5.93614578e-02  3.15370075e-02  6.00980893e-02 -5.29051982e-02
   4.06068079e-02 -2.59308275e-02  2.98428014e-02  1.12690229e-03
   7.35149086e-02 -5.03818654e-02 -1.22386590e-01  2.37028617e-02
   2.97265667e-02  4.24768180e-02  2.56337505e-02  1.99516583e-03
  -5.69190606e-02 -2.71598510e-02 -3.29035223e-02  6.60248846e-02
   1.19007163e-01 -4.58791703e-02 -7.26214275e-02 -3.25840227e-02
   5.23413271e-02  4.50552963e-02  8.25299788e-03  3.67023982e-02
  -1.39415748e-02  6.53918609e-02 -2.64272112e-02  2.06383236e-04
  -1.36643667e-02 -3.62810865e-02 -1.95044372e-02 -2.89738085e-02
   3.94270569e-02 -8.84090737e-02  2.62426562e-03  1.36713833e-02
   4.83062342e-02 -3.11566107e-02 -1.17329173e-01 -5.11690341e-02
  -8.85287970e-02 -2.18963120e-02  1.42986383e-02  4.44167443e-02
  -1.34815695e-02  7.43392035e-02  2.66382992e-02 -1.98762938e-02
   1.79191

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    chunks,
    embedding=model,
    persist_directory=persist_directory,
    collection_name="rag_collection"
    )

print(f"vectorstore created with {vectorstore._collection.count()} vectors")

C:\Users\mendh\AppData\Local\Temp\ipykernel_5852\842223121.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vectorstore created with 28 vectors


In [9]:
len(chunks)

28

### Similarity search

In [10]:
query = "What is Machine Learning?"

similar_chunks = vectorstore.similarity_search(query, k=3)
print("Similar Chunks:")
for i, chunk in enumerate(similar_chunks):
    print(f"Chunk {i+1}: {chunk.page_content[:200]}...")

Similar Chunks:
Chunk 1: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus...
Chunk 2: Machine learning (ML), reorganised and recognised as its own field, started to flourish in the 1990s. The field changed its goal from achieving artificial intelligence to tackling solvable problems of...
Chunk 3: Tom M. Mitchell provided a widely quoted, more formal definition of the algorithms studied in the machine learning field: "A computer program is said to learn from experience E with respect to some cl...


In [11]:
query = "What is Java?"

similar_chunks = vectorstore.similarity_search(query, k=3)
print("Similar Chunks:")
for i, chunk in enumerate(similar_chunks):
    print(f"Chunk {i+1}: {chunk.page_content[:200]}...")

Similar Chunks:
Chunk 1: Java is a versatile programming language. It is used for building enterprise applications, mobile apps, and large-scale systems....
Chunk 2: JavaScript is a popular programming language for web development. It is used to create interactive and dynamic web pages....
Chunk 3: Python is a high-level programming language. It is widely used for web development, data analysis, artificial intelligence, and scientific computing....


In [12]:
query = "What is Machine Learning?"

similar_chunks = vectorstore.similarity_search_with_relevance_scores(query, k=3)
print("Similar Chunks:")
for i, chunk in enumerate(similar_chunks):
    print(f"with relevance score: {chunk[1]:.4f}")
    print(f"Chunk {i+1}: {chunk[0].page_content[:200]}... ")

Similar Chunks:
with relevance score: 0.6298
Chunk 1: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus... 
with relevance score: 0.5586
Chunk 2: Machine learning (ML), reorganised and recognised as its own field, started to flourish in the 1990s. The field changed its goal from achieving artificial intelligence to tackling solvable problems of... 
with relevance score: 0.5560
Chunk 3: Tom M. Mitchell provided a widely quoted, more formal definition of the algorithms studied in the machine learning field: "A computer program is said to learn from experience E with respect to some cl... 


In [13]:
similar_chunks

[(Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.[1] Within a subdiscipline of machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.'),
  0.6297532120890769),
 (Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML), reorganised and recognised as its own field, started to flourish in the 1990s. The field changed its goal from achieving artificial intelligence to tackling solva

In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

llm.invoke("Hello, how are you?")

AIMessage(content="I'm just a computer program, so I don't have feelings, but thank you for asking. I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 41, 'total_tokens': 87, 'completion_time': 0.057952339, 'completion_tokens_details': None, 'prompt_time': 0.002558317, 'prompt_tokens_details': None, 'queue_time': 0.045699253, 'total_time': 0.060510656}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3f35-77ac-7113-9ad6-99f2399194cb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 46, 'total_tokens': 87})

In [15]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(model_provider="groq", model="llama-3.1-8b-instant")

llm.invoke("Hello, how are you?")

AIMessage(content="I'm a large language model, so I don't have emotions or feelings like humans do. However, I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 41, 'total_tokens': 88, 'completion_time': 0.131616643, 'completion_tokens_details': None, 'prompt_time': 0.002488617, 'prompt_tokens_details': None, 'queue_time': 0.045376813, 'total_time': 0.13410526}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3f35-799b-7e00-a288-fd17192e8e34-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 47, 'total_tokens': 88})

In [16]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain



In [17]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000024BFF732660>, search_kwargs={'k': 3})

In [18]:
from langchain_classic.prompts import ChatPromptTemplate

system_prompt = """
You are an assistant for question-answering tasks.
Use the following pieces of retrived conext to answer the question.
if you don't know the answer, say you don't know.

context: {context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")     
])


In [19]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks.\nUse the following pieces of retrived conext to answer the question.\nif you don't know the answer, say you don't know.\n\ncontext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [20]:
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks.\nUse the following pieces of retrived conext to answer the question.\nif you don't know the answer, say you don't know.\n\ncontext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outp

In [21]:
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000024BFF732660>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks.\nUse the following pieces of retrived conext to answer the question.\nif

In [22]:
response = rag_chain.invoke({"input": "What is Machine Learning?"})
response

{'input': 'What is Machine Learning?',
 'context': [Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.[1] Within a subdiscipline of machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.'),
  Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML), reorganised and recognised as its own field, started to flourish in the 1990s. The field changed its goal from achieving artificial int

In [23]:
print(response["answer"])

Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.


In [24]:
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_core.output_parsers import StrOutputParser

### Create RAG chain Alternative - using Lcel(LangChain Expression Language)

In [25]:
system_prompt = """Use the following context to answer the question. if you dontknow the answer based on the context, say you don't know. Provide specific details from the context to support your answer.
Context: {context}"""

custom_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [26]:
def format_retrieved_chunks(chunks: List[Document]) -> str:
    formatted_chunks = []
    for i, chunk in enumerate(chunks):
        formatted_chunks.append(f"Chunk {i+1}:\n{chunk.page_content}\n")
    return "\n".join(formatted_chunks)

In [27]:
rag_chain_lcel = ({
    "context": retriever|format_retrieved_chunks,
    "input": RunnablePassthrough()
}| custom_prompt| llm| StrOutputParser())

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000024BFF732660>, search_kwargs={'k': 3})
           | RunnableLambda(format_retrieved_chunks),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following context to answer the question. if you dontknow the answer based on the context, say you don't know. Provide specific details from the context to support your answer.\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 

In [28]:
response = rag_chain_lcel.invoke("What is Machine Learning?")
response

'Based on the provided context, Machine Learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data. These algorithms can perform tasks without explicit programming language instructions.'

### Add New Document to Existing Vector store

In [29]:
new_document = """Reinforcement learning is a type of machine learning where an agent learns to make decisions by performing actions in an environment to maximize cumulative reward. It involves an agent, environment, actions, and rewards. The agent takes actions based on a policy, receives feedback in the form of rewards, and updates its policy to improve future actions. Reinforcement learning is used in various applications such as robotics, game playing, and autonomous systems. 
"""

new_document = Document(page_content=new_document, metadata={"source": "manual input", "category": "machine learning"})

In [30]:
new_chunks = text_splitter.split_documents([new_document])

new_chunks

[Document(metadata={'source': 'manual input', 'category': 'machine learning'}, page_content='Reinforcement learning is a type of machine learning where an agent learns to make decisions by performing actions in an environment to maximize cumulative reward. It involves an agent, environment, actions, and rewards. The agent takes actions based on a policy, receives feedback in the form of rewards, and updates its policy to improve future actions. Reinforcement learning is used in various applications such as robotics, game playing, and autonomous systems.')]

In [31]:
vectorstore.add_documents(new_chunks)



['169d412a-a621-48c3-ab63-fe5b0e659e8e']

In [32]:
new_question = "What is Reinforcement Learning?"
response = rag_chain_lcel.invoke(new_question)
response

'Based on the given context, specifically Chunk 1, Reinforcement Learning is a type of machine learning where an agent learns to make decisions by performing actions in an environment to maximize cumulative reward. It involves an agent, environment, actions, and rewards. The agent takes actions based on a policy, receives feedback in the form of rewards, and updates its policy to improve future actions.'

### Conversational Memory

In [33]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage,AIMessage



In [34]:
contextualize_q_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history,formulate a standalone question
which can be understood without the chat history. Do not answer the question, just reformulate it if needed and otherwise return the question as is."""

contextualize_q_system_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}")
])

history_aware_retriver = create_history_aware_retriever(llm, retriever, contextualize_q_system_prompt)

In [35]:
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrived conext to answer the question.
if you don't know the answer, say you don't know.
Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)  

conversation_chain = create_retrieval_chain(history_aware_retriver, qa_chain)

In [36]:
chat_history = []
result = conversation_chain.invoke({"input": "What is Machine Learning?", "chat_history": chat_history})
result

{'input': 'What is Machine Learning?',
 'chat_history': [],
 'context': [Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.[1] Within a subdiscipline of machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.'),
  Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML), reorganised and recognised as its own field, started to flourish in the 1990s. The field changed its goal from ach

In [37]:
chat_history.extend([HumanMessage(content="What is Machine Learning?"), AIMessage(content=result["answer"])])

In [38]:
result2 = conversation_chain.invoke({"input": "What are its types?", "chat_history": chat_history})
result2

{'input': 'What are its types?',
 'chat_history': [HumanMessage(content='What is Machine Learning?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Modern-day Machine Learning algorithms are broken into 3 algorithm types: Supervised Learning Algorithms, Unsupervised Learning Algorithms, and Reinforcement Learning Algorithms.[17]'),
  Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, 